# 신약 후보물질 독성 예측 데이터 EDA

## 분석 목적

이 노트북은 `train.csv`의 데이터 구조와 품질을 확인하고 모델링 전에 주의해야 할 사항을 정리한다.

- 데이터 구조와 타입 확인
- 라벨 분포와 의미 확인
- 결측치·중복·이상값 점검
- SMILES 문자열 특성 확인
- ECFP·FCFP·PTFP 희소성 비교
- 상수 Fingerprint 탐색
- 물성값 분포와 라벨별 차이 확인
- 동일한 입력 특성에 서로 다른 라벨이 부여된 사례 탐색

> 이 노트북에서는 모델을 학습하지 않는다.

## 가장 중요한 라벨 정의

수업에서 제공한 데이터는 일반적인 독성 분류 관습과 반대로 라벨링되어 있다.

- `label = 1`: 독성 없음
- `label = 0`: 독성 있음

따라서 기본 F1 Score의 Positive 클래스는 **독성 없음**이다. 독성 물질 탐지 성능을 확인하려면 `label=0` 기준 Recall과 F1을 별도로 계산해야 한다.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

# 로컬 기본 경로. 다른 환경에서는 아래 경로만 수정한다.
DATA_PATH = Path("data/INPUT_REQUIRED")

if not DATA_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded)))
    except ImportError as exc:
        raise FileNotFoundError("DATA_PATH를 실제 train.csv 경로로 수정하세요.") from exc

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")

## 1. 데이터 기본 구조

In [ ]:
memory_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"행 수: {df.shape[0]:,}")
print(f"열 수: {df.shape[1]:,}")
print(f"메모리 사용량: {memory_mb:.1f} MB")
print("\n앞쪽 열:", df.columns[:10].tolist())
print("뒤쪽 열:", df.columns[-10:].tolist())
print("\n데이터 타입별 열 수")
display(df.dtypes.astype(str).value_counts().rename("column_count").to_frame())
display(df.head())

### 확인된 구조

- 8,349행, 3,078열
- `SMILES` 1개
- ECFP·FCFP·PTFP 각각 1,024개, 총 3,072개 Fingerprint
- `MolWt`, `clogp`, `sa_score`, `qed` 4개 Descriptor
- `label` 1개

In [ ]:
ecfp_cols = [c for c in df.columns if c.startswith("ecfp_")]
fcfp_cols = [c for c in df.columns if c.startswith("fcfp_")]
ptfp_cols = [c for c in df.columns if c.startswith("ptfp_")]
descriptor_cols = ["MolWt", "clogp", "sa_score", "qed"]
fingerprint_groups = {
    "ECFP": ecfp_cols,
    "FCFP": fcfp_cols,
    "PTFP": ptfp_cols,
}

structure_check = pd.DataFrame({
    "group": ["ECFP", "FCFP", "PTFP", "Descriptor"],
    "column_count": [len(ecfp_cols), len(fcfp_cols), len(ptfp_cols), len(descriptor_cols)],
})
display(structure_check)

## 2. 데이터 품질 점검

In [ ]:
quality_summary = pd.Series({
    "missing_cells": int(df.isna().sum().sum()),
    "columns_with_missing": int((df.isna().sum() > 0).sum()),
    "duplicated_rows": int(df.duplicated().sum()),
    "duplicated_smiles_extra": int(df["SMILES"].duplicated().sum()),
    "unique_smiles": int(df["SMILES"].nunique()),
    "invalid_labels": int((~df["label"].isin([0, 1])).sum()),
    "infinite_descriptors": int(np.isinf(df[descriptor_cols].to_numpy()).sum()),
}, name="count")

display(quality_summary.to_frame())

### 해석

- 결측치와 무한대 값이 없다.
- 완전히 중복된 행과 중복 SMILES가 없다.
- 라벨은 0과 1로 정상 구성되어 있다.

## 3. 라벨 분포

In [ ]:
label_summary = pd.DataFrame({
    "count": df["label"].value_counts().sort_index(),
    "ratio": df["label"].value_counts(normalize=True).sort_index(),
})
label_summary.index = ["0: 독성 있음", "1: 독성 없음"]
label_summary["ratio_percent"] = (label_summary["ratio"] * 100).round(1)
display(label_summary[["count", "ratio_percent"]])

try:
    import matplotlib.pyplot as plt

    ax = label_summary["count"].plot(kind="bar", color=["#d95f5f", "#5b8fd1"], figsize=(6, 4), rot=0)
    ax.set_title("Label Distribution")
    ax.set_ylabel("Count")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib이 없어 그래프를 생략했습니다.")

### 해석

- 독성 있음(0): 3,807건, 45.6%
- 독성 없음(1): 4,542건, 54.4%
- 심각한 클래스 불균형은 아니다.
- 기본 F1은 독성 없음(1)을 Positive로 계산한다는 점에 유의한다.

## 4. SMILES 분석

In [ ]:
smiles_length = df["SMILES"].str.len()

display(smiles_length.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame("SMILES_length"))

longest_smiles = df.loc[smiles_length.nlargest(10).index, ["SMILES", "label"]].copy()
longest_smiles["length"] = smiles_length.loc[longest_smiles.index]
display(longest_smiles.sort_values("length", ascending=False))

try:
    import matplotlib.pyplot as plt

    ax = smiles_length.plot(kind="hist", bins=40, figsize=(7, 4), color="#607d8b", edgecolor="white")
    ax.set_title("SMILES Length Distribution")
    ax.set_xlabel("Number of characters")
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

### 해석

- 평균 길이 56.6자, 중앙값 55자
- 최솟값 9자, 최댓값 249자
- 긴 SMILES는 복잡한 분자일 수 있으므로 길이만으로 제거하지 않는다.

## 5. Fingerprint 값과 희소성

In [ ]:
fingerprint_rows = []

for group_name, columns in fingerprint_groups.items():
    values = df[columns].to_numpy()
    row_active_bits = values.sum(axis=1)
    unique_values = np.unique(values)
    nunique = df[columns].nunique(dropna=False)

    fingerprint_rows.append({
        "group": group_name,
        "columns": len(columns),
        "unique_values": unique_values.tolist(),
        "nonzero_rate": values.mean(),
        "mean_active_bits": row_active_bits.mean(),
        "median_active_bits": np.median(row_active_bits),
        "min_active_bits": row_active_bits.min(),
        "max_active_bits": row_active_bits.max(),
        "constant_columns": int((nunique == 1).sum()),
    })

fingerprint_summary = pd.DataFrame(fingerprint_rows).set_index("group")
fingerprint_summary["nonzero_rate_percent"] = (fingerprint_summary["nonzero_rate"] * 100).round(2)
display(fingerprint_summary.drop(columns="nonzero_rate").round(2))

In [ ]:
active_bits_by_label = []

for group_name, columns in fingerprint_groups.items():
    active = df[columns].sum(axis=1)
    temp = pd.DataFrame({"label": df["label"], "active_bits": active})
    grouped = temp.groupby("label")["active_bits"].agg(["mean", "median", "std"])
    grouped.insert(0, "group", group_name)
    active_bits_by_label.append(grouped.reset_index())

active_bits_by_label = pd.concat(active_bits_by_label, ignore_index=True)
display(active_bits_by_label.round(2))

### 해석

- ECFP 활성 비율: 5.48%, 행당 평균 56.2비트
- FCFP 활성 비율: 4.39%, 행당 평균 44.9비트
- PTFP 활성 비율: 42.35%, 행당 평균 433.7비트
- ECFP와 FCFP는 매우 희소하고 PTFP는 상대적으로 조밀하다.
- 라벨별 활성 비트 수 차이는 작아서 활성 비트 개수만으로 독성을 구분하기 어렵다.

## 6. 상수 열 탐색

In [ ]:
all_feature_cols = ecfp_cols + fcfp_cols + ptfp_cols + descriptor_cols
constant_cols = [c for c in all_feature_cols if df[c].nunique(dropna=False) <= 1]

constant_summary = pd.DataFrame({
    "column": constant_cols,
    "constant_value": [df[c].iloc[0] for c in constant_cols],
})

print(f"상수 열 수: {len(constant_cols)}")
display(constant_summary)

### 해석

상수 열은 모두 PTFP에 속하며 총 49개다. 모든 행에서 값이 1이므로 라벨 구분 정보를 제공하지 않는다.

- 제거하면 입력 차원과 계산량을 조금 줄일 수 있다.
- 트리 모델은 보통 상수 열을 자동으로 사용하지 않으므로 성능 변화는 크지 않을 수 있다.
- 학습 데이터에서 제거했다면 예측 데이터에서도 동일한 열을 제거해야 한다.
- 이 열들이 화학적으로 항상 무의미한 것이 아니라 현재 데이터 범위에서만 분산이 0이라는 뜻이다.

## 7. Descriptor 분포

In [ ]:
descriptor_summary = df[descriptor_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T
display(descriptor_summary.round(4))

outlier_rows = []
for column in descriptor_cols:
    q1, q3 = df[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    mask = (df[column] < q1 - 1.5 * iqr) | (df[column] > q3 + 1.5 * iqr)
    outlier_rows.append({
        "column": column,
        "iqr_outliers": int(mask.sum()),
        "ratio": mask.mean(),
    })

outlier_summary = pd.DataFrame(outlier_rows).set_index("column")
outlier_summary["ratio_percent"] = (outlier_summary.pop("ratio") * 100).round(2)
display(outlier_summary)

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for column, ax in zip(descriptor_cols, axes.ravel()):
        for label, color, name in [(0, "#d95f5f", "Toxic"), (1, "#5b8fd1", "Non-toxic")]:
            df.loc[df["label"] == label, column].plot(
                kind="hist", bins=35, alpha=0.45, density=True,
                color=color, label=name, ax=ax
            )
        ax.set_title(column)
        ax.legend()
    fig.suptitle("Descriptor Distributions by Label", y=1.02)
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

### 이상치 해석

- MolWt: 130건(1.56%)
- clogp: 137건(1.64%)
- sa_score: 158건(1.89%)
- qed: 0건

화학 데이터에서는 큰 분자량이나 높은 합성 난이도가 실제 특성일 수 있으므로 IQR 이상치를 자동 삭제하지 않는다.

## 8. 라벨별 Descriptor 차이

In [ ]:
group_descriptor = df.groupby("label")[descriptor_cols].agg(["mean", "median", "std"])
display(group_descriptor.round(4))

correlation = df[descriptor_cols + ["label"]].corr()
display(correlation.round(4))

effect_rows = []
for column in descriptor_cols:
    toxic = df.loc[df["label"] == 0, column]
    non_toxic = df.loc[df["label"] == 1, column]
    pooled_std = np.sqrt(
        ((len(toxic) - 1) * toxic.var() + (len(non_toxic) - 1) * non_toxic.var())
        / (len(toxic) + len(non_toxic) - 2)
    )
    effect_rows.append({
        "column": column,
        "toxic_mean_label_0": toxic.mean(),
        "non_toxic_mean_label_1": non_toxic.mean(),
        "cohen_d_label1_minus_label0": (non_toxic.mean() - toxic.mean()) / pooled_std,
    })

display(pd.DataFrame(effect_rows).set_index("column").round(4))

### 해석

- `clogp` 평균은 독성 그룹 4.21, 무독성 그룹 3.45로 가장 큰 차이를 보였다.
- `clogp`의 Cohen's d는 약 -0.57로 중간 정도의 차이다.
- `MolWt`는 두 그룹 평균이 거의 같아 단독 구분력이 낮아 보인다.
- `sa_score`와 `qed`의 차이는 작다.
- 상관관계는 인과관계를 의미하지 않으며 단일 변수만으로 독성을 결정할 수 없다.

## 9. Fingerprint별 라벨 비율 차이가 큰 비트

In [ ]:
prevalence_rows = []

for group_name, columns in fingerprint_groups.items():
    toxic_rate = df.loc[df["label"] == 0, columns].mean()
    non_toxic_rate = df.loc[df["label"] == 1, columns].mean()
    diff = (non_toxic_rate - toxic_rate).abs().sort_values(ascending=False).head(10)

    for column, abs_difference in diff.items():
        prevalence_rows.append({
            "group": group_name,
            "column": column,
            "toxic_prevalence": toxic_rate[column],
            "non_toxic_prevalence": non_toxic_rate[column],
            "absolute_difference": abs_difference,
        })

top_prevalence_difference = pd.DataFrame(prevalence_rows)
for column in ["toxic_prevalence", "non_toxic_prevalence", "absolute_difference"]:
    top_prevalence_difference[column + "_percent"] = (top_prevalence_difference[column] * 100).round(2)
display(top_prevalence_difference.drop(columns=[
    "toxic_prevalence", "non_toxic_prevalence", "absolute_difference"
]))

### 해석

일부 Fingerprint 비트는 두 라벨에서 출현 비율 차이가 크다. 다만 수천 개 열을 동시에 비교했기 때문에 우연한 차이와 과적합 가능성이 있다. 실제 모델링에서는 교차검증을 통해 재현되는지 확인해야 한다.

## 10. 동일 입력 특성과 라벨 충돌

In [ ]:
# SMILES와 label을 제외한 모든 입력 특성이 같은 행을 같은 그룹으로 본다.
model_feature_cols = ecfp_cols + fcfp_cols + ptfp_cols + descriptor_cols
feature_hash = pd.util.hash_pandas_object(df[model_feature_cols], index=False)

feature_groups = pd.DataFrame({
    "feature_hash": feature_hash.astype(str),
    "SMILES": df["SMILES"],
    "label": df["label"],
})

group_stats = feature_groups.groupby("feature_hash").agg(
    rows=("label", "size"),
    label_count=("label", "nunique"),
)

duplicate_feature_groups = group_stats[group_stats["rows"] > 1]
conflicting_group_ids = group_stats[group_stats["label_count"] > 1].index
conflicting_rows = feature_groups[
    feature_groups["feature_hash"].isin(conflicting_group_ids)
].copy()

print(f"동일 입력 특성 그룹: {len(duplicate_feature_groups):,}개")
print(f"해당 그룹의 행: {int(duplicate_feature_groups['rows'].sum()):,}개")
print(f"서로 다른 라벨을 가진 그룹: {len(conflicting_group_ids):,}개")
print(f"라벨 충돌 그룹의 행: {len(conflicting_rows):,}개")
print(f"최대 그룹 크기: {int(group_stats['rows'].max())}개")

In [ ]:
# 그룹 번호를 붙여 전체 SMILES와 라벨을 확인한다.
group_number_map = {group_id: i + 1 for i, group_id in enumerate(sorted(conflicting_group_ids))}
conflicting_rows["conflict_group"] = conflicting_rows["feature_hash"].map(group_number_map)

display(
    conflicting_rows[["conflict_group", "SMILES", "label"]]
    .sort_values(["conflict_group", "label"])
    .reset_index(drop=True)
)

### 해석

- 동일 입력 특성 그룹: 241개
- 해당 행: 534개
- 상반된 라벨을 포함하는 그룹: 33개
- 라벨 충돌 그룹의 행: 73개
- 최대 동일 특성 그룹 크기: 5개

충돌 사례 대부분은 SMILES의 `@`와 `@@`가 다른 입체이성질체다. 현재 Fingerprint와 4개 Descriptor가 입체화학 차이를 충분히 표현하지 못하지만 실제 독성은 달라질 수 있다. 따라서 이를 단순 라벨 오류라고 단정하면 안 된다.

무작위 분할 시 거의 동일한 특성의 행이 학습과 검증 데이터에 동시에 들어가 성능이 과대평가될 수 있으므로, 모델링 단계에서는 동일 특성 그룹을 같은 fold에 배치하는 Group 기반 검증을 고려한다.

## 11. 최종 EDA 결론

1. 데이터에는 결측치, 무한대 값, 중복 SMILES가 없다.
2. 라벨 비율은 45.6% 대 54.4%로 심각한 불균형은 아니다.
3. `label=1`이 독성 없음이므로 기본 F1의 의미를 주의해야 한다.
4. ECFP와 FCFP는 매우 희소하고 PTFP는 상대적으로 조밀하다.
5. PTFP 상수 열 49개는 현재 데이터에서 예측 정보를 제공하지 않아 제거 후보이다.
6. Descriptor 중 `clogp`가 라벨과 가장 뚜렷한 차이를 보인다.
7. 동일 입력 특성에 상반된 라벨이 있는 33개 그룹이 존재하며 대부분 입체화학 차이와 관련된다.
8. 모델링에서는 상수 열 제거, 동일 전처리 적용, Stratified 검증과 Group 기반 검증 비교가 필요하다.
9. 독성 탐지 관점의 Recall과 F1을 기본 F1과 별도로 확인해야 한다.